In [9]:
# %pip install Biopython
# %pip install BOLDigger3
#!pip install boldigger3
# !pip install pygbif requests pandas

In [26]:
"""
GenBank Data Integration Module
Integrates molecular data from GenBank into occurrence records with geographic filtering
"""

import pandas as pd
import time
import re
from Bio import Entrez, SeqIO
from math import radians, sin, cos, sqrt, asin
from typing import Optional, Dict, List, Tuple

# ============================================================================
# NAME STANDARDIZATION
# ============================================================================

ABBREVIATION_MAP = {
    'P.': 'Phlebotomus', 'Ph.': 'Phlebotomus', 'Se.': 'Sergentomyia',
    'L.': 'Leishmania', 'Aa.': 'Aedes', 'An.': 'Anopheles',
    'Cx.': 'Culex', 'Cs.': 'Culiseta', 'Ps.': 'Psorophora',
    'Ha.': 'Haemagogus', 'Sa.': 'Sabethes', 'Wy.': 'Wyeomyia',
    'Tr.': 'Triatoma', 'Rh.': 'Rhodnius', 'Pa.': 'Panstrongylus',
    'Tb.': 'Tabanus', 'St.': 'Stomoxys', 'Gl.': 'Glossina',
    'Mu.': 'Musca', 'Lu.': 'Lutzomyia', 'Br.': 'Brugia',
    'Wu.': 'Wuchereria', 'Lo.': 'Loa', 'On.': 'Onchocerca'
}

def fix_spacing(name: str) -> str:
    """Fix spacing issues in species names."""
    name = ' '.join(name.split())
    name = re.sub(r'\.([a-zA-Z])', r'. \1', name)
    name = re.sub(r'(\w+)\s+\.', r'\1.', name)
    return ' '.join(name.split())

def parse_genus_species(name: str) -> Optional[Tuple[str, str]]:
    """Split species name into genus and species parts."""
    parts = name.split()
    if len(parts) < 2:
        return None
    genus = parts[0]
    epithet = ' '.join(parts[1:]).lower()
    return genus, epithet

def normalize_case(genus: str, epithet: str) -> str:
    """Standardize capitalization of genus and species."""
    if '.' not in genus and len(genus) > 2:
        genus = genus.capitalize()
    return f"{genus} {epithet}"

def clean_species_name(species_name: str) -> Optional[str]:
    """Clean and standardize species name."""
    if not species_name or pd.isna(species_name):
        return None
    
    fixed = fix_spacing(species_name)
    parsed = parse_genus_species(fixed)
    
    if parsed is None:
        return None
    
    genus, epithet = parsed
    return normalize_case(genus, epithet)

def expand_abbreviation(species_name: str) -> Optional[str]:
    """Expand genus abbreviation to full name."""
    cleaned = clean_species_name(species_name)
    if cleaned is None:
        return None
    
    parts = cleaned.split()
    if len(parts) < 2:
        return cleaned
    
    genus = parts[0]
    epithet = ' '.join(parts[1:])
    
    if genus in ABBREVIATION_MAP:
        return f"{ABBREVIATION_MAP[genus]} {epithet}"
    
    return cleaned

# ============================================================================
# COORDINATE UTILITIES
# ============================================================================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate great-circle distance in kilometers."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * asin(sqrt(a)) * EARTH_RADIUS_KM

def parse_coordinates(coord_str: str) -> Optional[Tuple[float, float]]:
    """Parse coordinate string (DMS or decimal)."""
    if 'N' in coord_str or 'S' in coord_str:
        lat_match = re.search(r'(\d+\.?\d*)\s*([NS])', coord_str)
        lon_match = re.search(r'(\d+\.?\d*)\s*([EW])', coord_str)
        if lat_match and lon_match:
            lat = float(lat_match.group(1))
            lon = float(lon_match.group(1))
            if lat_match.group(2) == 'S': lat = -lat
            if lon_match.group(2) == 'W': lon = -lon
            return lat, lon
    else:
        coords = coord_str.split()
        if len(coords) >= 2:
            try:
                return float(coords[0]), float(coords[1])
            except ValueError:
                pass
    return None

# ============================================================================
# GENBANK DATA EXTRACTION
# ============================================================================

def extract_gene(seq_record) -> Optional[str]:
    """Extract gene name from GenBank record."""
    for feature in seq_record.features:
        if feature.type == "gene" and 'gene' in feature.qualifiers:
            return feature.qualifiers['gene'][0]
    return None

def extract_country(feature) -> Tuple[Optional[str], Optional[str]]:
    """Extract country and location from feature."""
    country = location = None
    if 'country' in feature.qualifiers:
        parts = feature.qualifiers['country'][0].split(':')
        country = parts[0].strip()
        if len(parts) > 1:
            location = parts[1].strip()
    return country, location

def extract_geo_from_feature(feature) -> Tuple[Optional[float], Optional[float]]:
    """Extract coordinates from feature."""
    if 'lat_lon' not in feature.qualifiers:
        return None, None
    coords = feature.qualifiers['lat_lon'][0]
    parsed = parse_coordinates(coords)
    if parsed:
        return parsed
    return None, None

def extract_record_metadata(seq_record, species_name: str) -> Dict:
    """Extract metadata from GenBank record."""
    return {
        'gb_accession': seq_record.id,
        'gb_sequence_length': len(seq_record.seq),
        'gb_definition': seq_record.description,
        'gb_organism': seq_record.annotations.get('organism', species_name),
        'gb_taxonomy': '; '.join(seq_record.annotations.get('taxonomy', [])),
        'gb_gene': extract_gene(seq_record)
    }

def extract_geographic_info(seq_record) -> Dict:
    """Extract geographic information from GenBank record."""
    geo = {'gb_lat': None, 'gb_lon': None, 'gb_country': None, 'gb_location': None}
    
    for feature in seq_record.features:
        if feature.type != "source":
            continue
        
        country, location = extract_country(feature)
        geo['gb_country'] = country
        geo['gb_location'] = location
        
        lat, lon = extract_geo_from_feature(feature)
        geo['gb_lat'] = lat
        geo['gb_lon'] = lon
        
        if 'collection_date' in feature.qualifiers:
            geo['gb_collection_date'] = feature.qualifiers['collection_date'][0]
        break
    
    if 'location' in seq_record.annotations and not geo['gb_location']:
        geo['gb_location'] = seq_record.annotations['location']
    
    return geo

# ============================================================================
# GENBANK QUERIES
# ============================================================================

def search_genbank(species_name: str, retmax: int = SEARCH_LIMIT) -> List[str]:
    """Search GenBank for species and return ID list."""
    Entrez.email = NCBI_EMAIL
    handle = Entrez.esearch(db="nucleotide", term=f'"{species_name}"[Organism]', retmax=retmax)
    record = Entrez.read(handle)
    handle.close()
    return record.get("IdList", [])

def process_records_from_handle(handle, species_name: str, target_lat: float, target_lon: float, 
                                radius_km: float, max_records: int) -> List[Dict]:
    """Process GenBank records from an open handle."""
    candidates = []
    records = SeqIO.parse(handle, "genbank")
    
    for seq_record in records:
        geo = extract_geographic_info(seq_record)
        
        if geo['gb_lat'] is None or geo['gb_lon'] is None:
            continue
        
        distance = haversine_distance(target_lat, target_lon, geo['gb_lat'], geo['gb_lon'])
        if distance > radius_km:
            continue
        
        metadata = extract_record_metadata(seq_record, species_name)
        candidates.append({
            **metadata,
            'gb_country': geo.get('gb_country'),
            'gb_location': geo.get('gb_location'),
            'gb_lat': geo['gb_lat'],
            'gb_lon': geo['gb_lon'],
            'gb_distance_km': round(distance, 2),
            'gb_collection_date': geo.get('gb_collection_date')
        })
    
    candidates.sort(key=lambda x: x['gb_distance_km'])
    return candidates[:max_records]

def get_geographically_filtered_records(
    species_name: str,
    target_lat: float,
    target_lon: float,
    radius_km: float = DEFAULT_RADIUS_KM,
    max_records: int = DEFAULT_MAX_RECORDS
) -> Optional[List[Dict]]:
    """Get GenBank records filtered by geographic proximity."""
    try:
        id_list = search_genbank(species_name)
        if not id_list:
            return None
        
        handle = Entrez.efetch(db="nucleotide", id=id_list[:SEARCH_LIMIT], rettype="gb", retmode="text")
        candidates = process_records_from_handle(handle, species_name, target_lat, target_lon, 
                                                 radius_km, max_records)
        handle.close()
        
        if candidates:
            return candidates
        
        print(f"  ⚠️ No close records found. Using fallback...")
        return get_fallback_records(species_name, max_records)
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None

def process_fallback_handle(handle, species_name: str, max_records: int) -> List[Dict]:
    """Process fallback records from an open handle."""
    candidates = []
    records = SeqIO.parse(handle, "genbank")
    
    for seq_record in records:
        metadata = extract_record_metadata(seq_record, species_name)
        metadata.update({
            'gb_country': None, 'gb_location': None,
            'gb_lat': None, 'gb_lon': None,
            'gb_distance_km': None, 'gb_collection_date': None
        })
        candidates.append(metadata)
    
    return candidates[:max_records]

def get_fallback_records(species_name: str, max_records: int = 3) -> Optional[List[Dict]]:
    """Fallback: get any records without geographic filtering."""
    try:
        id_list = search_genbank(species_name, retmax=max_records)
        if not id_list:
            return None
        
        handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="gb", retmode="text")
        candidates = process_fallback_handle(handle, species_name, max_records)
        handle.close()
        return candidates
    
    except Exception as e:
        print(f"  ✗ Fallback error: {e}")
        return None

# ============================================================================
# DATAFRAME OPERATIONS
# ============================================================================

def normalize_species_names(df: pd.DataFrame, species_col: str = 'species') -> pd.DataFrame:
    """Add cleaned and expanded species name columns."""
    df = df.copy()
    df['original_species'] = df[species_col]
    df['species_cleaned'] = df[species_col].apply(clean_species_name)
    df['species_full_name'] = df['species_cleaned'].apply(expand_abbreviation)
    
    total = len(df)
    valid = df['species_full_name'].notna().sum()
    
    print(f"Name normalization: {valid}/{total} ({valid/total*100:.1f}%) successful")
    return df

def filter_valid_species(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with invalid species names."""
    valid = df['species_full_name'].notna()
    invalid_count = (~valid).sum()
    
    if invalid_count:
        print(f"⚠️ Skipping {invalid_count} rows with invalid species names")
        invalid_names = df[~valid]['species'].unique()[:5]
        for name in invalid_names:
            print(f"   - '{name}'")
    
    return df[valid].copy()

def aggregate_species_coordinates(df: pd.DataFrame, lat_col: str, lon_col: str, 
                                  species_col: str = 'species') -> pd.DataFrame:
    """Group species and calculate mean coordinates."""
    return df.groupby('species_full_name').agg({
        lat_col: 'mean',
        lon_col: 'mean',
        species_col: 'first',
        'species_cleaned': 'first',
        'original_species': 'first'
    }).reset_index()

def enrich_record_with_context(record: Dict, species_full: str, original: str, cleaned: str, 
                               target_lat: float, target_lon: float, idx: int) -> Dict:
    """Add linking information to GenBank record."""
    record.update({
        'species_full_name': species_full,
        'original_species': original,
        'cleaned_species': cleaned,
        'gb_record_number': idx + 1,
        'gb_target_lat': target_lat,
        'gb_target_lon': target_lon,
        'gb_is_closest': idx == 0
    })
    return record

def process_species_group(row: pd.Series, lat_col: str, lon_col: str, 
                          radius: float, max_records: int) -> List[Dict]:
    """Process one species group and fetch GenBank records."""
    species_full = row['species_full_name']
    target_lat = row[lat_col]
    target_lon = row[lon_col]
    
    print(f"\nFetching: {species_full} at ({target_lat:.4f}, {target_lon:.4f})")
    
    records = get_geographically_filtered_records(
        species_full, target_lat, target_lon, radius, max_records
    )
    
    if not records:
        print("  ✗ No records found")
        return []
    
    enriched = []
    for i, record in enumerate(records):
        enriched.append(enrich_record_with_context(
            record, species_full, row['original_species'], row['species_cleaned'],
            target_lat, target_lon, i
        ))
    
    closest = records[0].get('gb_distance_km', 'unknown')
    print(f"  ✓ Found {len(records)} records (closest: {closest} km)")
    return enriched

# ============================================================================
# COLUMN MANAGEMENT
# ============================================================================

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Rename id column to rec_id if present."""
    if 'id' in df.columns:
        df = df.rename(columns={'id': 'rec_id'})
    return df

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def merge_with_genbank(
    df: pd.DataFrame,
    species_col: str = 'species',
    lat_col: str = 'Lat',
    lon_col: str = 'Lon',
    radius_km: float = DEFAULT_RADIUS_KM,
    max_records: int = DEFAULT_MAX_RECORDS,
    output_file: Optional[str] = None
) -> pd.DataFrame:
    """
    Integrate GenBank data into occurrence records with geographic filtering.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input data with species and coordinates
    species_col : str
        Column name for species
    lat_col, lon_col : str
        Column names for coordinates
    radius_km : float
        Search radius in kilometers
    max_records : int
        Maximum GenBank records per species
    output_file : str, optional
        Output CSV filename
    
    Returns
    -------
    pd.DataFrame
        Merged data with GenBank information (all GenBank columns prefixed with 'gb_')
    """
    # Clean column names
    df = clean_column_names(df)
    
    # Normalize and filter species
    df = normalize_species_names(df, species_col)
    df = filter_valid_species(df)
    
    # Prepare species groups
    species_groups = aggregate_species_coordinates(df, lat_col, lon_col, species_col)
    print(f"Processing {len(species_groups)} unique species...")
    
    # Fetch GenBank records
    all_records = []
    for _, row in species_groups.iterrows():
        records = process_species_group(row, lat_col, lon_col, radius_km, max_records)
        all_records.extend(records)
        time.sleep(1)  # NCBI rate limit
    
    if not all_records:
        print("No GenBank data retrieved.")
        return df
    
    # Convert to DataFrame (all columns already have gb_ prefix)
    genbank_df = pd.DataFrame(all_records)
    
    # Merge results
    merged_df = df.merge(genbank_df, on=['original_species', 'species_full_name'], how='left')
    
    print(f"\n✓ Retrieved {len(genbank_df)} records from {genbank_df['species_full_name'].nunique()} species")
    print(f"  Merged: {len(merged_df)} rows, {len(merged_df.columns)} columns")
    
    # Show GenBank columns
    gb_cols = [col for col in merged_df.columns if col.startswith('gb_')]
    print(f"  GenBank columns added: {len(gb_cols)}")
    
    if output_file:
        merged_df.to_csv(output_file, index=False)
        print(f"  Saved to: {output_file}")
    
    return merged_df



In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

NCBI_EMAIL = "your_email@domain.com"
EARTH_RADIUS_KM = 6371
DEFAULT_RADIUS_KM = 500
DEFAULT_MAX_RECORDS = 3
SEARCH_LIMIT = 50

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

# Load your data
base_data = pd.read_csv('your_data.csv')  # Replace with actual file

# Run integration
merged = merge_with_genbank(
    base_data,
    species_col='species',
    lat_col='Lat', # Latitude field
    lon_col='Lon', # Longitude field
    radius_km=500, # For distance evaluation
    max_records=3, # To be retrieve from Genbank
    output_file='dashboard_data_with_genbank.csv'
)

# Show column summary
print("\n📊 Column Summary:")
original_cols = [c for c in merged.columns if not c.startswith('gb_')]
gb_cols = [c for c in merged.columns if c.startswith('gb_')]

print(f"  Total columns: {len(merged.columns)}")
print(f"  Original columns ({len(original_cols)}): {', '.join(original_cols[:8])}...")
print(f"  GenBank columns ({len(gb_cols)}): {', '.join(gb_cols[:8])}...")

# Show summary of species with GenBank data
print("\n📊 Species with GenBank data:")
summary = merged[merged['gb_accession'].notna()].groupby(
    ['original_species', 'species_full_name']
).size().reset_index(name='gb_records')
print(summary.head(10).to_string(index=False))